In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import pandas as pd 
from utils_data_preproc import merge_datasets,create_combined_rel_abund_and_metadata

In [18]:
combined = create_combined_rel_abund_and_metadata(
    train_rel_path="./data/CMD - Healthy subjects/curatedMD_healthy_train/20221118_healthy_taxrelabund_train.csv",
    test_rel_path="./data/CMD - Healthy subjects/curatedMD_healthy_test/20221118_healthy_taxrelabund_test.csv",
    train_meta_path="./data/CMD - Healthy subjects/curatedMD_healthy_train/20221118_healthy_metadata_train.csv",
    test_meta_path="./data/CMD - Healthy subjects/curatedMD_healthy_test/20221118_healthy_metadata_test_nlytics.csv",
    out_rel_path="./data/healthy_tax_relabund_all.csv",
    out_meta_path="./data/healthy_metadata_all.csv",
)

rel_all = combined.rel_abund
meta_all = combined.metadata


In [19]:
df_CMD_all = merge_datasets(
    path_otu="./data/healthy_tax_relabund_all.csv",
    path_metadata="./data/healthy_metadata_all.csv",
    level=7,          # e.g., genus depending on the format
    how="inner",
)

In [20]:
print(f"Number of Samples: {len(df_CMD_all)}")
print(f"Number of Columns: {len(df_CMD_all.columns)}")

Number of Samples: 3873
Number of Columns: 1052


In [6]:
df_CMD_all_filt = df_CMD_all[
    # Sequencing quality filters
    (df_CMD_all["number_reads"] >= 1_000_000) &
    (df_CMD_all["median_read_length"] >= 90) &

    # Age-related filters
    (df_CMD_all["age_category"].str.lower() == "adult") &
    (df_CMD_all["age"].notna())
].copy()

In [7]:

METADATA_COLS = [
    "study_name", "sample_id", "subject_id", "body_site",
    "antibiotics_current_use", "study_condition", "disease", "age",
    "age_category", "country", "non_westernized", "sequencing_platform",
    "PMID", "number_reads", "number_bases", "minimum_read_length",
    "median_read_length", "curator"
]

def filter_species_features(
    df_CMD_all: pd.DataFrame,
    *,
    metadata_cols=METADATA_COLS,
    prevalence_min: float = 0.05,
    mean_abundance_min: float = 1e-5,
    variance_min: float | None = None,
    presence_threshold: float = 0.0,
) -> pd.DataFrame:
    """
    Filter species/features in a combined (metadata + species) dataframe.

    Filters (common in microbiome analyses; see refs below):
      - prevalence: keep features present (> presence_threshold) in >= prevalence_min fraction of samples
      - mean abundance: keep features with mean >= mean_abundance_min
      - optional variance: keep features with variance >= variance_min

    Parameters
    ----------
    prevalence_min : float
        Fraction of samples a feature must be present in (e.g., 0.05 = 5%).
        Prevalence filtering is widely used to reduce sparsity and false associations
        (e.g., McMurdie & Holmes 2014; Weiss et al. 2017; large-scale gut studies like Falony et al. 2016).
    mean_abundance_min : float
        Minimum mean relative abundance to keep (e.g., 1e-5).
        Abundance filters are common to remove ultra-low signal features that are often noise-dominated.
    variance_min : float | None
        Optional; if set, removes near-constant features (use only if still too many features).
    presence_threshold : float
        What counts as "present". For relative abundance tables, 0.0 is typical.
    """
    df = df_CMD_all.copy()

    # --- 1) Split metadata vs species columns ---
    missing_meta = [c for c in metadata_cols if c not in df.columns]
    if missing_meta:
        raise ValueError(f"These metadata columns are missing from df_CMD_all: {missing_meta}")

    species_cols = [c for c in df.columns if c not in metadata_cols]

    if len(species_cols) == 0:
        raise ValueError("No species columns detected. Check metadata_cols or dataframe columns.")

    # --- 2) Make species matrix numeric ---
    X = df[species_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)

    # --- 3) Prevalence filter (most important in sparse microbiome data) ---
    # Present if > presence_threshold
    prevalence = (X > presence_threshold).mean(axis=0)
    keep_prev = prevalence >= prevalence_min

    # --- 4) Mean abundance filter ---
    means = X.mean(axis=0)
    keep_mean = means >= mean_abundance_min

    keep = keep_prev & keep_mean

    # --- 5) Optional variance filter ---
    if variance_min is not None:
        variances = X.var(axis=0)
        keep_var = variances >= variance_min
        keep = keep & keep_var

    kept_species_cols = list(X.columns[keep])

    # --- 6) Recombine metadata + filtered species ---
    df_filtered = pd.concat([df[metadata_cols], X[kept_species_cols]], axis=1)

    return df_filtered


In [8]:
import numpy as np
import pandas as pd

# scikit-bio transforms
from skbio.stats.composition import ilr, clr


def prepare_logratio_for_lingam(
    df_CMD_all: pd.DataFrame,
    metadata_cols: list[str],
    *,
    transform: str = "ilr",                # {"ilr", "clr", "alr"}
    reads_col: str = "number_reads",
    pseudocount: float | None = None,
    alr_ref: str | None = None,            # required if transform="alr"
    standardize: bool = True,
    drop_clr_col: str | None = None,       # optional: drop one clr component to avoid singularity
) -> pd.DataFrame:
    """
    Prepare species data from a CMD-like dataframe (metadata + species relative abundances)
    for LiNGAM using a selectable log-ratio transform: ILR, CLR, or ALR.

    Pipeline (common to all transforms)
    ----------------------------------
    1) Extract species columns (all non-metadata columns).
    2) Convert relative abundances to pseudo-counts by multiplying by read depth.
       NOTE: these are pseudo-counts (RA * depth), not raw mapping counts.
    3) Add a pseudocount to ensure strictly positive values.
    4) Apply chosen transform:
       - ILR: scikit-bio ilr -> returns (D-1) balances (full rank).
       - CLR: scikit-bio clr -> returns D components with per-row sum-to-zero (singular covariance).
              Optionally drop one component (recommended for LiNGAM).
       - ALR: log(x_i/x_ref) for each species i != ref -> returns (D-1) variables (full rank).
    5) (Optional) Standardize (z-score) columns for numerical stability in LiNGAM.

    Parameters
    ----------
    df_CMD_all : pd.DataFrame
        DataFrame containing metadata + species relative abundances.
    metadata_cols : list[str]
        Columns to treat as metadata (kept out of transformation).
    transform : str
        One of {"ilr", "clr", "alr"} (case-insensitive).
    reads_col : str
        Column containing sequencing depth (default: "number_reads").
    pseudocount : float | None
        Added to pseudo-count matrix to avoid zeros. If None, uses half of the smallest
        positive pseudo-count observed in the table.
    alr_ref : str | None
        Reference species column name used as denominator in ALR. Required if transform="alr".
        Must be one of the species columns.
    standardize : bool
        If True, z-score each transformed feature column (recommended for LiNGAM).
    drop_clr_col : str | None
        If transform="clr": drop this CLR component to remove rank deficiency.
        If None, drops the last CLR column.

    Returns
    -------
    pd.DataFrame
        Transformed feature matrix ready for LiNGAM:
          - ILR/ALR: shape (n_samples, D-1)
          - CLR: shape (n_samples, D) unless drop_clr_col is used (then D-1)

    Raises
    ------
    ValueError
        If required columns are missing or transform arguments are invalid.
    """

    # -----------------------------
    # 1) Sanity checks
    # -----------------------------
    if reads_col not in df_CMD_all.columns:
        raise ValueError(f"'{reads_col}' not found in df_CMD_all columns.")

    missing_meta = [c for c in metadata_cols if c not in df_CMD_all.columns]
    if missing_meta:
        raise ValueError(f"Missing metadata columns: {missing_meta}")

    transform = transform.lower().strip()
    if transform not in {"ilr", "clr", "alr"}:
        raise ValueError("transform must be one of {'ilr', 'clr', 'alr'}.")

    # -----------------------------
    # 2) Extract species matrix
    # -----------------------------
    species_cols = [c for c in df_CMD_all.columns if c not in metadata_cols]
    if len(species_cols) < 2:
        raise ValueError(
            f"Need at least 2 species columns for log-ratio transforms; found {len(species_cols)}."
        )

    X_rel = (
        df_CMD_all[species_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .astype(float)
    )

    # -----------------------------
    # 3) Convert to pseudo-counts: RA * read depth
    # -----------------------------
    reads = df_CMD_all[reads_col].astype(float).to_numpy().reshape(-1, 1)

    # Guard against missing/invalid reads
    if np.any(~np.isfinite(reads)):
        raise ValueError(f"Non-finite values found in '{reads_col}'.")
    if np.any(reads <= 0):
        raise ValueError(f"All values in '{reads_col}' must be > 0 to scale pseudo-counts.")

    X_counts = X_rel.to_numpy() * reads  # shape: (n_samples, D)

    # -----------------------------
    # 4) Add pseudocount (zeros handling)
    # -----------------------------
    if pseudocount is None:
        positive = X_counts[X_counts > 0]
        if positive.size == 0:
            raise ValueError("No positive entries after pseudo-count construction.")
        pseudocount = 0.5 * float(positive.min())

    if not np.isfinite(pseudocount) or pseudocount <= 0:
        raise ValueError("pseudocount must be a positive finite number.")

    X_pos = X_counts + pseudocount  # strictly positive

    # -----------------------------
    # 5) Apply transformation
    # -----------------------------
    if transform == "ilr":
        Z = ilr(X_pos)  # returns (n_samples, D-1)
        cols = [f"ilr_{i+1}" for i in range(Z.shape[1])]
        Z_df = pd.DataFrame(Z, index=df_CMD_all.index, columns=cols)

    elif transform == "clr":
        Z = clr(X_pos)  # returns (n_samples, D)
        cols = [f"clr_{c}" for c in species_cols]
        Z_df = pd.DataFrame(Z, index=df_CMD_all.index, columns=cols)

        # CLR has a sum-to-zero constraint -> singular covariance.
        # For LiNGAM it is typically better to drop one component (D-1 dims).
        if drop_clr_col is None:
            drop_clr_col = Z_df.columns[-1]
        if drop_clr_col not in Z_df.columns:
            raise ValueError(f"drop_clr_col='{drop_clr_col}' not found among CLR columns.")
        Z_df = Z_df.drop(columns=[drop_clr_col])

    else:  # transform == "alr"
        if alr_ref is None:
            raise ValueError("alr_ref must be provided when transform='alr'.")

        if alr_ref not in species_cols:
            raise ValueError(
                f"alr_ref='{alr_ref}' must be one of the species columns. "
                f"Example species cols: {species_cols[:5]}..."
            )

        # ALR: log(x_i/x_ref) = log(x_i) - log(x_ref)
        logX = np.log(X_pos)
        ref_idx = species_cols.index(alr_ref)
        Z = logX - logX[:, [ref_idx]]  # subtract reference column from all columns
        # remove the reference column itself (it becomes all zeros)
        Z = np.delete(Z, ref_idx, axis=1)

        cols = [f"alr_{c}_vs_{alr_ref}" for c in species_cols if c != alr_ref]
        Z_df = pd.DataFrame(Z, index=df_CMD_all.index, columns=cols)

    # -----------------------------
    # 6) (Optional) Standardize for LiNGAM stability
    # -----------------------------
    if standardize:
        # ddof=0 for population std; avoid division by zero
        mu = Z_df.mean(axis=0)
        sd = Z_df.std(axis=0, ddof=0).replace(0.0, np.nan)
        Z_df = (Z_df - mu) / sd
        Z_df = Z_df.fillna(0.0)

    return Z_df


In [58]:
 # transformation please check https://www.geo.fu-berlin.de/en/v/soga-r/Advances-statistics/Feature-scales/Logratio_Transformations/index.html

In [12]:
df_CMD_all_species_filt = filter_species_features(
    df_CMD_all_filt,
    prevalence_min=0.3,        # 5% prevalence
    mean_abundance_min=1e-4,    # remove ultra-low mean abundance
    variance_min=None          # set e.g. 1e-6 only if still too many species
    
)

In [13]:
X_lingam = prepare_logratio_for_lingam(
    df_CMD_all_species_filt,
    metadata_cols=METADATA_COLS,
    transform="clr",
    standardize=False
)

In [14]:
X_lingam.to_csv("./exp4/data/clr_transformation_cmd_healthy_no_standardization_30percent_presence.csv", index= None)

In [15]:
X_lingam

,clr_Adlercreutzia_equolifaciens,clr_Agathobaculum_butyriciproducens,clr_Akkermansia_muciniphila,clr_Alistipes_finegoldii,clr_Alistipes_indistinctus,clr_Alistipes_putredinis,clr_Alistipes_shahii,clr_Anaeromassilibacillus_sp_An250,clr_Anaerostipes_hadrus,clr_Anaerotruncus_colihominis,...,clr_Ruminococcus_lactaris,clr_Ruminococcus_torques,clr_Ruthenibacterium_lactatiformans,clr_Slackia_isoflavoniconvertens,clr_Streptococcus_parasanguinis,clr_Streptococcus_salivarius,clr_Streptococcus_thermophilus,clr_Turicibacter_sanguinis,clr_Turicimonas_muris,clr_Veillonella_dispar
0,-5.464890,3.066374,-5.464890,2.505772,-5.464890,5.147224,-5.464890,-5.464890,1.667912,-3.061302,...,3.399377,3.559675,1.905144,1.644100,-0.726882,2.703447,0.053188,0.236523,-5.464890,-5.464890
1,-1.290349,2.516752,5.128229,2.421766,-0.329438,4.689847,2.772765,1.420548,1.345095,-6.053658,...,-6.053658,3.374222,0.636684,-6.053658,-0.560118,1.589196,0.790473,-3.417372,-6.053658,-6.053658
2,-0.165846,3.459044,2.675797,3.740706,1.075613,4.422126,0.008892,-2.374964,2.370562,-6.415169,...,1.947436,2.876105,0.463376,-0.654284,-0.540451,-0.029667,-2.330786,1.169906,-1.248328,-1.315173
3,1.447211,4.588163,-5.882334,1.635711,-5.882334,4.590539,3.579751,-5.882334,4.197855,-5.882334,...,2.344517,2.635034,-0.219288,2.325768,-5.882334,-2.414129,-5.882334,-5.882334,-1.955084,-1.502623
4,0.631569,2.333622,4.831999,0.314348,-0.723819,5.597243,2.553661,-2.448725,3.683351,1.529562,...,2.621451,2.989213,1.971545,-6.197172,0.164792,0.864316,0.733321,-6.197172,-6.197172,-6.197172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3079,-4.741097,-4.741097,-4.741097,2.175635,-4.741097,6.098331,0.569671,-4.741097,5.362756,-4.741097,...,-4.741097,-1.179017,1.507875,-4.741097,2.454304,3.082465,-4.741097,-4.741097,3.762704,2.873125
3080,-6.240819,1.692924,-6.240819,0.729836,-2.910329,4.683489,2.627466,-6.240819,3.474785,-6.240819,...,0.142114,2.790861,0.701557,-6.240819,-1.357217,-0.229499,-0.471520,0.264076,0.944780,-0.439245
3081,-3.175038,-3.175038,-3.175038,-3.175038,-3.175038,4.264987,-3.175038,-3.175038,7.738313,-3.175038,...,-3.175038,-0.506592,-3.175038,-3.175038,-3.175038,2.715491,-3.175038,4.801574,6.016269,-3.175038
3082,-1.794829,0.531164,-5.535031,-5.535031,1.664778,6.249498,2.497922,-5.535031,2.074393,-0.422691,...,-5.535031,3.638465,0.756189,-5.535031,-5.535031,-0.418890,-5.535031,1.626758,-5.535031,-5.535031


In [154]:
pd.read_csv("./exp3/data/clr_transformation_cmd_healthy.csv")

,clr_Acidaminococcus_intestini,clr_Adlercreutzia_equolifaciens,clr_Agathobaculum_butyriciproducens,clr_Akkermansia_muciniphila,clr_Alistipes_finegoldii,clr_Alistipes_indistinctus,clr_Alistipes_inops,clr_Alistipes_onderdonkii,clr_Alistipes_putredinis,clr_Alistipes_shahii,...,clr_Sellimonas_intestinalis,clr_Slackia_isoflavoniconvertens,clr_Streptococcus_parasanguinis,clr_Streptococcus_salivarius,clr_Streptococcus_thermophilus,clr_Turicibacter_sanguinis,clr_Turicimonas_muris,clr_Veillonella_atypica,clr_Veillonella_dispar,clr_Veillonella_parvula
0,-0.239453,-1.342673,0.416392,-1.058851,0.441468,-0.962522,1.105506,-0.319536,0.582008,-1.626149,...,2.825054,1.151716,0.348309,0.845453,0.657435,1.156064,-0.653663,-0.551368,-0.588744,0.408557
1,3.348852,-0.027895,0.127034,1.191500,0.380634,0.434106,1.626793,-0.617524,0.447504,0.660234,...,-0.561629,-0.887244,0.352678,0.430948,0.817513,-0.062090,-0.901330,-0.811116,-0.850087,-0.994765
2,-0.484761,0.482000,0.694222,0.756664,0.830717,0.949924,1.713353,-0.587965,0.484383,0.001638,...,-0.531722,0.628042,0.497788,0.043770,0.081487,1.544971,0.848219,1.928269,0.972516,1.503770
3,-0.433717,0.880629,0.997022,-1.172920,0.189535,-1.110864,-0.692384,-0.532110,0.434726,0.900139,...,-0.475207,1.299730,-1.332390,-0.864918,-0.999560,-0.837611,0.480797,0.641215,0.767753,-0.931955
4,-0.419712,0.729369,0.213991,1.212640,-0.058948,0.434084,0.659975,-0.516785,0.734317,0.709781,...,-0.459701,-0.822112,0.709805,0.322825,0.909895,-0.825403,-0.817603,-0.723305,-0.761736,-0.920685
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2646,-0.192679,-1.301014,-3.044187,-1.031385,0.203727,-0.926804,-0.522601,-0.268353,0.660170,-0.088188,...,-0.208337,-0.661487,1.173089,0.774362,-0.819642,-0.627505,2.227924,2.057433,2.138561,1.736086
2647,-0.479581,-1.556537,-0.099023,-1.199850,0.017017,-0.212305,-0.724690,-0.582297,0.506665,0.696874,...,-0.525986,-0.864468,0.186038,-0.073999,0.547032,1.203851,1.526905,0.807674,1.224647,1.171916
2648,0.102677,-1.037963,-2.712870,-0.857957,-1.364637,-0.701268,-0.314558,0.054840,0.088434,-1.362571,...,0.118672,-0.452525,-0.866137,0.405279,-0.599180,2.182307,2.726069,1.550264,-0.260407,2.084266
2649,-0.356223,-0.223851,-0.729674,-1.127416,-1.685411,0.966514,1.016760,-0.447312,0.770247,0.555671,...,1.410661,-0.777193,-1.265029,-0.256876,-0.941716,1.521571,-0.759861,-0.662747,-0.700807,-0.869597


In [ ]:
X_lingam = prepare_logratio_for_lingam(
    df_CMD_all_species_filt,
    metadata_cols=METADATA_COLS,
    transform="ilr",
    standardize=False
)

In [50]:
prevalence =(df_CMD_all_species_filt.drop(columns=METADATA_COLS) > 0).sum(axis=0)/len(df_CMD_all_species_filt)

In [55]:
prevalence.sort_values(ascending=False)

Faecalibacterium_prausnitzii       0.989815
Blautia_wexlerae                   0.925311
Dorea_longicatena                  0.924180
Eubacterium_rectale                0.922293
Fusicatenibacter_saccharivorans    0.921162
                                     ...   
Firmicutes_bacterium_CAG_791       0.051301
Citrobacter_youngae                0.050924
Actinomyces_sp_oral_taxon_181      0.050924
Enterococcus_faecalis              0.050170
Prevotella_timonensis              0.050170
Length: 270, dtype: float64

In [164]:
X_lingam = prepare_logratio_for_lingam(
    df_CMD_all_species_filt,
    metadata_cols=METADATA_COLS,
    transform="alr",
    alr_ref="Eubacterium_hallii",  # example; must exist as a species column
    standardize=True
)

In [169]:
X_lingam.to_csv("alr_transformation_cmd_healthy.csv", index=None)

In [116]:
import matplotlib.pyplot as plt
import numpy as np
# 1. Calculate correlation
corr = X_lingam.corr()

corr.corr().style.background_gradient(cmap='coolwarm', axis=None).to_html('clr_correlation_alr.html')

In [ ]:
# pick-up a feature using prevalence and variance. 
prevalence = (df_CMD_all_species_filt.drop(columns=METADATA_COLS) > 0).sum(axis=0) / len(df_CMD_all_species_filt)

prevalence_species = pd.DataFrame(prevalence.sort_values(ascending=False))

variance_species = pd.DataFrame(df_CMD_all_species_filt.drop(columns=METADATA_COLS).var().sort_values(ascending=False))
df_join_variance_prevalence = variance_species.join(prevalence_species, lsuffix="l_")
df_join_variance_prevalence = df_join_variance_prevalence.rename(columns={'0l_':"variance", '0':'prevalence'})
df_join_variance_prevalence[(df_join_variance_prevalence['variance'] < 1) & (df_join_variance_prevalence['prevalence'] > 0.9)]

,0
Faecalibacterium_prausnitzii,0.989815
Blautia_wexlerae,0.925311
Dorea_longicatena,0.924180
Eubacterium_rectale,0.922293
Fusicatenibacter_saccharivorans,0.921162
...,...
Firmicutes_bacterium_CAG_791,0.051301
Citrobacter_youngae,0.050924
Actinomyces_sp_oral_taxon_181,0.050924
Enterococcus_faecalis,0.050170


In [121]:
data = np.load("../adjacency_model_run_time.npz", allow_pickle=True)
A = data["adjacency"]
features = data["features"].tolist()
adj_df = pd.DataFrame(A, index=features, columns=features)


In [123]:
adj_df.shape

(20, 20)